<span style="font-size:2em; font-weight:bold;">Objective 2: Evaluation of Ground Motion Spatial Correlation in New Zealand Through Empirical Approaches</span>
***

This code was written to replicate and extend the work done by Bodenmann et al., 2023 (https://doi.org/10.5194/nhess-23-2387-2023) which included parameters for estimating path and site effects within spatial ground-motion correlations models. The method also utilized Bayesian Inference for the fitting of Spatial Correlation models in lieu of the traditional weight-least squares methods.

# 1.0 Import Modules

## 1.1 Basic

In [ ]:
# Repository paths (works when Jupyter starts at the repository root or in notebooks/).
from pathlib import Path
import sys

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (parent for parent in (_current, *_current.parents)
     if (parent / "src" / "cbb21_cossim.py").is_file()
     and (parent / "data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found. Start Jupyter in the repository or notebooks/ directory.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = REPO_ROOT / "data"
RESIDUALS_DIR = DATA_DIR / "residuals"
SCM_DIR = DATA_DIR / "nzwide_scm"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
import seaborn as sns
import random
import xarray as xr
from scipy.stats import invgamma
from IPython.display import clear_output

## 1.2 Numpyro/JAX

In [ ]:
import arviz as az
import numpyro as numpyro
print('numpyro v' + str(numpyro.__version__))
import jax as jax
print('jax v' + str(jax.__version__))

# Change JAX default from float32 to float64
jax.config.update("jax_enable_x64", True)

# Import some functions directly for convenience
import jax.numpy as jnp
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive, log_likelihood
from scipy.special import logsumexp

numpyro.set_platform("cpu")

# Number of CPUs -> To sample several MCMC chains in parallel
numpyro.set_host_device_count(4)

numpyro.enable_validation(True)

# 2.0 JAX Function and Model Definitions

## 2.1 Dissimilarity Metrics Functions

In [ ]:
def getEucDistanceFromPolar(X):
    sq_dist = (jnp.square(jnp.reshape(X[:,0],[-1,1])) +
               jnp.square(jnp.reshape(X[:,0],[1,-1])) -
        2 * (jnp.reshape(X[:,0],[-1,1]) * jnp.reshape(X[:,0],[1,-1])) *
        jnp.cos(jnp.abs(jnp.reshape(X[:,1],[1,-1]) - jnp.reshape(X[:,1],[-1,1]) )) )
    sq_dist = jnp.clip(sq_dist, 0, jnp.inf)
    dist = jnp.sqrt(sq_dist)
    dist = dist.at[jnp.diag_indices_from(dist)].set(0.0)
    return dist

def getAngDistanceFromPolar(X):
    cos_angle = jnp.cos( jnp.abs(jnp.reshape(X[:,1],[1,-1]) -
                                jnp.reshape(X[:,1],[-1,1])))
    dist =  jnp.arccos(jnp.clip(cos_angle, -1, 1))
    return dist

def getSoilDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,2], [-1,1]) - jnp.reshape(X[:,2], [1,-1]))
    sq_dist = jnp.clip(sq_dist, 0, jnp.inf)
    dist = jnp.sqrt(sq_dist)
    return dist

def getBasinDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,3],[-1,1])-jnp.reshape(X[:,3],[1,-1]))
    sq_dist = jnp.clip(sq_dist,0,jnp.inf)
    distZ = jnp.sqrt(sq_dist)
    return distZ

def getTopoDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,4],[-1,1])-jnp.reshape(X[:,4],[1,-1]))
    sq_dist = jnp.clip(sq_dist,0,jnp.inf)
    distT = jnp.sqrt(sq_dist)
    return distT

def getTWIDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,5], [-1,1]) - jnp.reshape(X[:,5], [1,-1]))
    sq_dist = jnp.clip(sq_dist,0,jnp.inf)
    distTWI = jnp.sqrt(sq_dist)
    return distTWI

def getGeoDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,6], [-1,1]) - jnp.reshape(X[:,6], [1,-1]))
    sq_dist = jnp.clip(sq_dist,0,jnp.inf)
    distGEO = jnp.sqrt(sq_dist)
    return distGEO

def getPcpDissimilarity(X):
    sq_dist = jnp.square(jnp.reshape(X[:,7], [-1,1]) - jnp.reshape(X[:,7], [1,-1]))
    sq_dist = jnp.clip(sq_dist,0,jnp.inf)
    distPCP = jnp.sqrt(sq_dist)
    return distPCP

## 2.2 Model definitions and Prior distributions

In [ ]:
# Define distributions (scipy uses shape=a, scale=β)
dists = {
    "InvGamma(α=2, β=30)": invgamma(a=3, scale=25),
    "InvGamma(α=2, β=100)": invgamma(a=2, scale=100),
}

x = np.linspace(0.001, 500, 1000)  # adjust range as needed

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi = 200)

for ax, (label, d) in zip(axes, dists.items()):
    pdf_vals = d.pdf(x)
    ax.plot(x, pdf_vals, label="PDF")

    # Mean
    mean_val = d.mean()
    ax.axvline(mean_val, color="red", linestyle="--", label=f"Mean = {mean_val:.3f}")

    # 5th and 95th percentiles
    q05, q95 = d.ppf([0.05, 0.95])
    ax.axvline(q05, color="green", linestyle=":", label=f"5% = {q05:.3f}")
    ax.axvline(q95, color="blue", linestyle=":", label=f"95% = {q95:.3f}")

    ax.set_title(label)
    ax.set_xlabel("x")
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

### 2.2.1 Euclidean Distance Model (Model E)

Isotropic model based on Euclidean distance (Model E):

$$
\rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}}) = \exp(- (d_{\mathrm{E}}/\ell_{\mathrm{E}})^{\gamma_{\mathrm{E}}})
$$

In [ ]:
# Deterministic correlation functions
def rhoE(X, LEt, gammaE, nugget=1e-6):
    distE = getEucDistanceFromPolar(X)
    K = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

# Probabilistic correlation model
def modelE(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoE(X[i], LEt, gammaE)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.2 Angular Distance Model (Model EA)

Accounting for path effects using angular distance (Model EA)
$$
\rho_{\mathrm{EA}}(d_{\mathrm{E}}, d_{\mathrm{A}}; \boldsymbol{\psi}_{\mathrm{EA}}) = \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}}) \cdot \rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})
$$
where
$$
\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}}) = (1 + d_{\mathrm{A}}/\ell_{\mathrm{A}})(1 - d_{\mathrm{A}}/180)^{180/\ell_{\mathrm{A}}}
$$

As mentioned in the main study the prior is defined on a transformed version of the angular lengthscale, $\ell_{\mathrm{A}}$, to respect its domain $(0°,45°)$. The transformed version is specified as `LAt` in the code below.

In [ ]:
def rhoEA(X, LEt, gammaE, LA, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    K = KE * KA
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEA(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEA(X[i], LEt, gammaE, LA)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.3 Soil Dissimilarity Model (Model EAS)

Accounting for site effects using soil dissimilarity (Model EAS)
$$
\rho_{\mathrm{EAS}}(d_{\mathrm{E}}, d_{\mathrm{A}}, , d_{\mathrm{S}}; \boldsymbol{\psi}_{\mathrm{EAS}}) = \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}}) \cdot (w \rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}}) + (1-w) \rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}}))
$$
where
$$
\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}}) = \exp(-d_{\mathrm{S}}/\ell_{\mathrm{S}})
$$

In [ ]:
def rhoEAS(X, LEt, gammaE, LA, LS, w, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    K = KE * (w * KA + (1-w) * KS)
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEAS(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2, 100))
    w = numpyro.sample("w", dist.Beta(2,2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEAS(X[i], LEt, gammaE, LA, LS, w)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.4 Basin Dissimilarity Model (Model EASZ)

Accounting for site effects using soil and basin dissimilarity (Model EASZ)
$$
\rho_{\mathrm{EASZ}}(d_{\mathrm{E}}, d_{\mathrm{A}}, d_{\mathrm{S}}, d_{\mathrm{Z1.0}}; \boldsymbol{\psi}_{\mathrm{PEAS}})= \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}})\cdot \Big( w \,\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})+ (1 - w)\,\big[ w_{\mathrm{S}} \,\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}})+ (1 - w_{\mathrm{S}})\,\rho_{\mathrm{Z}}(d_{\mathrm{Z1.0}}; \ell_{\mathrm{Z}}) \big] \Big)
$$
where
$$
\rho_{\mathrm{Z}}(d_{\mathrm{Z1.0}}; \ell_{\mathrm{Z}}) = \exp(-d_{\mathrm{Z1.0}}/\ell_{\mathrm{Z}})
$$

In [ ]:
def rhoEASZ(X, LEt, gammaE, LA, LS, LZ, w, wS, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    distZ = getBasinDissimilarity(X)
    KZ = jnp.exp(-1.0 * jnp.multiply(distZ, 1.0/LZ))

    KS_w = KS * wS
    KZ_w = KZ * (1-wS)

    K = KE * (w * KA + (1-w) * (KS_w + KZ_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEASZ(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2, 100))
    LZ = numpyro.sample("LZ", dist.InverseGamma(2, 100))
    w = numpyro.sample("w", dist.Beta(2,2))
    wS = numpyro.sample("wS", dist.Beta(2,2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEASZ(X[i], LEt, gammaE, LA, LS, LZ, w, wS)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.5 Topographic Dissimilarity Model (Model EAST)

Accounting for site effects using soil and topographic dissimilarity (Model EAST)
$$
\rho_{\mathrm{EAST}}(d_{\mathrm{E}}, d_{\mathrm{A}}, d_{\mathrm{S}}, d_{\mathrm{TRI}}; \boldsymbol{\psi}_{\mathrm{PEAS}})= \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}})\cdot \Big( w \,\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})+ (1 - w)\,\big[ w_{\mathrm{S}} \,\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}})+ (1 - w_{\mathrm{S}})\,\rho_{\mathrm{T}}(d_{\mathrm{TRI}}; \ell_{\mathrm{T}}) \big] \Big)
$$
where
$$
\rho_{\mathrm{T}}(d_{\mathrm{TRI}}; \ell_{\mathrm{T}}) = \exp(-d_{\mathrm{TRI}}/\ell_{\mathrm{T}})
$$


In [ ]:
def rhoEAST(X, LEt, gammaE, LA, LS, LT, w, wS, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    distT = getTopoDissimilarity(X)
    KT = jnp.exp(-1.0 * jnp.multiply(distT, 1.0/LT))

    KS_w = KS * wS
    KT_w = KT * (1-wS)

    K = KE * (w * KA + (1 - w) * (KS_w + KT_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEAST(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2, 100))
    LT = numpyro.sample("LT", dist.InverseGamma(2, 30))

    w = numpyro.sample("w", dist.Beta(2,2))
    wS = numpyro.sample("wS", dist.Beta(2,2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEAST(X[i], LEt, gammaE, LA, LS, LT, w, wS)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.6 Precipitation Dissimilarity Model (Model EASP)

Accounting for site effects using soil and precipitation dissimilarity (Model EASP)
$$
\rho_{\mathrm{EASP}}(d_{\mathrm{E}}, d_{\mathrm{A}}, d_{\mathrm{S}}, d_{\mathrm{PCP}}; \boldsymbol{\psi}_{\mathrm{PEAS}})= \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}})\cdot \Big( w \,\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})+ (1 - w)\,\big[ w_{\mathrm{S}} \,\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}})+ (1 - w_{\mathrm{S}})\,\rho_{\mathrm{P}}(d_{\mathrm{PCP}}; \ell_{\mathrm{P}}) \big] \Big)
$$
where
$$
\rho_{\mathrm{P}}(d_{\mathrm{PCP}}; \ell_{\mathrm{P}}) = \exp(-d_{\mathrm{PCP}}/\ell_{\mathrm{P}})
$$


In [ ]:
def rhoEASP(X, LEt, gammaE, LA, LS, LT, w, wS, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    distP = getPcpDissimilarity(X)
    KP = jnp.exp(-1.0 * jnp.multiply(distP, 1.0/LT))

    KS_w = KS * wS
    KP_w = KP * (1-wS)

    K = KE * (w * KA + (1 - w) * (KS_w + KP_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEASP(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2, 100))
    LP = numpyro.sample("LP", dist.InverseGamma(3, 25))

    w = numpyro.sample("w", dist.Beta(2,2))
    wS = numpyro.sample("wS", dist.Beta(2,2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEASP(X[i], LEt, gammaE, LA, LS, LP, w, wS)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.7 Topographic Wetness Index  Dissimilarity Model (Model EASTWI)

Accounting for site effects using soil and topographic wetness index dissimilarity (Model EASTWI)
$$
\rho_{\mathrm{EASTWI}}(d_{\mathrm{E}}, d_{\mathrm{A}}, d_{\mathrm{S}}, d_{\mathrm{TWI}}; \boldsymbol{\psi}_{\mathrm{PEAS}})= \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}})\cdot \Big( w \,\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})+ (1 - w)\,\big[ w_{\mathrm{S}} \,\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}})+ (1 - w_{\mathrm{S}})\,\rho_{\mathrm{TWI}}(d_{\mathrm{TWI}}; \ell_{\mathrm{P}}) \big] \Big)
$$
where
$$
\rho_{\mathrm{TWI}}(d_{\mathrm{TWI}}; \ell_{\mathrm{P}}) = \exp(-d_{\mathrm{TWI}}/\ell_{\mathrm{P}})
$$

In [ ]:
def rhoEASTWI(X, LEt, gammaE, LA, LS, LTWI, w, wS, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    distTWI = getTWIDissimilarity(X)
    KTWI = jnp.exp(-1.0 * jnp.multiply(distTWI, 1.0/LTWI))

    KTWI_w = KTWI * (1-wS)
    KS_w = KS * wS

    K = KE * (w * KA + (1 - w) * (KTWI_w +  KS_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEASTWI(X, eqids, z):
    # Priors
    LE = numpyro.sample("LE", dist.InverseGamma(2, 30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2,100))
    LTWI = numpyro.sample("LTWI", dist.InverseGamma(3,9))

    # Weights
    w = numpyro.sample("w", dist.Beta(2,2))
    wS = numpyro.sample("wS", dist.Beta(2,2))

    # Transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0 * gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))

    LA = numpyro.deterministic("LA", 180 / (4.0 + LAt))

    # Likelihood
    z = [numpyro.sample(f"z_{eqid}",
        dist.MultivariateNormal(0, rhoEASTWI(X[i], LEt, gammaE, LA, LS, LTWI, w, wS)),
        obs=z[i]) for i, eqid in enumerate(eqids)]

### 2.2.8 Topographic and Basin Dissimilarity Model (Model EASZT)

Accounting for site effects using soil, topographic and basin dissimilarity (Model EASZT)
$$
\rho_{\mathrm{EASTZ}}(d_{\mathrm{E}}, d_{\mathrm{A}}, d_{\mathrm{S}}, d_{\mathrm{Z1.0}}, d_{\mathrm{TRI}}; \boldsymbol{\psi}_{\mathrm{EASTZ}})
= \rho_{\mathrm{E}}(d_{\mathrm{E}}; \boldsymbol{\psi}_{\mathrm{E}})
\cdot \Bigg(
    w \,\rho_{\mathrm{A}}(d_{\mathrm{A}}; \ell_{\mathrm{A}})
    + (1 - w)\,\Big[
        w_{\mathrm{S}} \,\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}})
        + w_{\mathrm{Z}} \,\rho_{\mathrm{Z}}(d_{\mathrm{Z1.0}}; \ell_{\mathrm{Z}})
        + w_{\mathrm{T}} \,\rho_{\mathrm{T}}(d_{\mathrm{TRI}}; \ell_{\mathrm{T}})
    \Big]
\Bigg)
$$

where

$$
\rho_{\mathrm{S}}(d_{\mathrm{S}}; \ell_{\mathrm{S}}) = \exp\!\left(-\tfrac{d_{\mathrm{S}}}{\ell_{\mathrm{S}}}\right), \quad
\rho_{\mathrm{Z}}(d_{\mathrm{Z1.0}}; \ell_{\mathrm{Z}}) = \exp\!\left(-\tfrac{d_{\mathrm{Z1.0}}}{\ell_{\mathrm{Z}}}\right), \quad
\rho_{\mathrm{T}}(d_{\mathrm{TRI}}; \ell_{\mathrm{T}}) = \exp\!\left(-\tfrac{d_{\mathrm{TRI}}}{\ell_{\mathrm{T}}}\right).
$$

In [ ]:
def rhoEASZT(X, LEt, gammaE, LA, LS, LZ, LT, w, wS, wZ, wT, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distS = getSoilDissimilarity(X)
    KS = jnp.exp(-1.0 * jnp.multiply(distS, 1.0/LS))

    distZ = getBasinDissimilarity(X)
    KZ = jnp.exp(-1.0 * jnp.multiply(distZ, 1.0/LZ))

    distT = getTopoDissimilarity(X)
    KT = jnp.exp(-1.0 * jnp.multiply(distT, 1.0/LT))

    KS_w = KS * wS
    KZ_w = KZ * wZ
    KT_w = KT * wT

    K = KE * (w * KA + (1 - w) * (KS_w + KZ_w + KT_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEASZT(X, eqids, z):
    # Define Prior Distributions
    LE = numpyro.sample("LE", dist.InverseGamma(concentration=2, rate=30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LS = numpyro.sample("LS", dist.InverseGamma(2, 100))
    LZ = numpyro.sample("LZ", dist.InverseGamma(2, 100))
    LT = numpyro.sample("LT", dist.InverseGamma(2, 30))
    w = numpyro.sample("w", dist.Beta(2,2))

    u1 = numpyro.sample("u1", dist.Beta(2, 2))
    u2 = numpyro.sample("u2", dist.Beta(2, 2))

    wS = numpyro.deterministic("wS", u1)
    wZ = numpyro.deterministic("wZ", (1.0 - u1) * u2)
    wT = numpyro.deterministic("wT", (1.0 - u1) * (1.0 - u2))

    # Compute transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0*gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))
    LA = numpyro.deterministic("LA", 180/(4.0 + LAt))

    # Specify observational model for each event
    z = [numpyro.sample("z_{}".format(eqid),
              dist.MultivariateNormal(0, rhoEASZT(X[i], LEt, gammaE, LA, LS, LZ, LT, w, wS, wZ, wT)),
              obs = z[i]) for i,eqid in enumerate(eqids)]

### 2.2.9 Topographic Wetness Index and Geological Age Dissimilarity Model (Model EATWIGEO)

In [ ]:
def rhoEATWIGEO(X, LEt, gammaE, LA, LGEO, LTWI, w, wGEO, wTWI, nugget=1e-6):

    distE = getEucDistanceFromPolar(X)
    KE = jnp.exp(-1.0 * jnp.multiply(jnp.power(distE, gammaE), 1.0/LEt))

    distA = getAngDistanceFromPolar(X)
    distAdeg = jnp.multiply(distA, 180.0/np.pi)
    KA = ((1 + jnp.multiply(distAdeg, 1.0/LA)) *
          jnp.power(1 - jnp.multiply(distAdeg, 1.0/180), 180/LA))

    distTWI = getTWIDissimilarity(X)
    KTWI = jnp.exp(-1.0 * jnp.multiply(distTWI, 1.0/LTWI))

    distGEO = getGeoDissimilarity(X)
    KGEO = jnp.exp(-1.0 * jnp.multiply(distGEO, 1.0/LGEO))

    KGEO_w = KGEO * wGEO
    KTWI_w = KTWI * wTWI

    K = KE * (w * KA + (1 - w) * (KGEO_w + KTWI_w))
    K = K.at[jnp.diag_indices(X.shape[0])].add(nugget)
    return K

def modelEATWIGEO(X, eqids, z):
    # Priors
    LE = numpyro.sample("LE", dist.InverseGamma(2, 30))
    gamma2 = numpyro.sample("gamma2", dist.Beta(2,2))
    LAt = numpyro.sample("LAt", dist.Gamma(2, 0.25))
    LGEO = numpyro.sample("LGEO", dist.InverseGamma(3, 100))
    LTWI = numpyro.sample("LTWI", dist.InverseGamma(3, 9))

    # Weights
    w = numpyro.sample("w", dist.Beta(2,2))

    u1 = numpyro.sample("u1", dist.Beta(2, 2))

    wGEO = numpyro.deterministic("wGEO", u1)
    wTWI = numpyro.deterministic("wTWI", (1.0 - u1))

    # Transformed parameters
    gammaE = numpyro.deterministic("gammaE", 2.0 * gamma2)
    LEt = numpyro.deterministic("LEt", jnp.power(LE, gammaE))

    LA = numpyro.deterministic("LA", 180 / (4.0 + LAt))

    # Likelihood
    z = [numpyro.sample(f"z_{eqid}",
        dist.MultivariateNormal(0, rhoEATWIGEO(X[i], LEt, gammaE, LA, LGEO, LTWI, w, wGEO, wTWI)),
        obs=z[i]) for i, eqid in enumerate(eqids)]

# 3.0 Pareto-smoothed importance-sampling leave-one-out cross-validation (PSIS-LOOCV) Computation

For each observation i, we approximate the log predictive density the model would have assigned if that observation had been left out of fitting. PSIS uses importance weights derived from the log-likelihood draws to avoid refitting the model n times.

Outputs:
- ELPD LOO: Expected log predictive density over all observations (higher is better).
- p_loo: Effective number of parameters (complexity penalty).
- LOO: Often reported as -2 * elpd_loo (deviance scale) for comparisons.
- Pointwise contributions: If pointwise=True, per-observation elpd values and diagnostics (e.g., Pareto- k estimates) help identify problematic observations.
- Pareto- k diagnostics:
Large k (> 0.7–1.0) flags unstable importance weights, meaning the LOO approximation for those observations is unreliable and may require refitting or model adjustments.


In [ ]:
def compute_pooled_loo(idata, pointwise=True):
    # Concatenate all log_likelihood variables
    log_lik_vars = [idata.log_likelihood[v] for v in idata.log_likelihood.data_vars]
    log_lik_concat = xr.concat(log_lik_vars, dim="obs_id")

    # Create a copy of idata with the replaced log_likelihood
    idata_copy = idata.copy()
    idata_copy.log_likelihood = xr.Dataset({"log_lik": log_lik_concat})

    # Compute LOO
    loo_result = az.loo(idata_copy, var_name="log_lik", pointwise=pointwise)
    return loo_result

# 4.0 Model Selection Analysis

In this section, we evaluate several models using a 60/40 train–test split. The dataset includes only events with more than 40 observations and is truncated at a maximum vibrational period of 6s. To ensure consistency in model selection across different period ranges, the event set is defined according to this maximum period threshold.

In [ ]:
# output directory
input_file = RESIDUALS_DIR / "NZGMDB_v4p3_dwbar_T6.csv"
export_directory = SCM_DIR
export_directory.mkdir(parents=True, exist_ok=True)
im_dwbar_file = input_file.stem

# load data
im_dwbar = pd.read_csv(input_file, dtype={"sta": "string"})
evid_40 = im_dwbar['evid'].unique().tolist()

# specify periods
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0']

# create random subset for train/test
train_percent = 0.60
random.seed(50)
randn_evid = random.sample(range(len(evid_40)),round(len(evid_40)*train_percent))
rand_evid = [evid_40[i] for i in randn_evid]
testn_evid = list(set(range(len(evid_40))) - set(randn_evid))
test_evid = [evid_40[i] for i in testn_evid]

# Define random number and models
rng = np.random.default_rng(31)
model_name = ['modelE', 'modelEA', 'modelEAS', 'modelEASZ', 'modelEAST','modelEASP','modelEASTWI','modelEASZT','modelEATWIGEO']

# begin Bayesian Inference
pooled_summary_all = []
period_summaries = []

for period in pSAobs_T:
    clear_output(wait=True)
    print(f"Period: {period}")

    # Specify evids from pooled dataset
    eqids = rand_evid

    # create training dataset
    Xtot = []; ztot = []
    for eqid in eqids:
      group = im_dwbar[im_dwbar.evid==eqid].copy()
      Xtot.append(group[['epicenter_dist','epicenter_azimuth','vs30','Z1.0','TRI','TWI','geoage','precip']].values)
      ztot.append(group[period].values)
    train_data = dict(X=Xtot, eqids=eqids, z=ztot)

    # generate new seed per period
    seeds = rng.integers(0, 100000, 9)

    results = []
    predictions = []
    lppds = []
    models = [modelE, modelEA, modelEAS, modelEASZ, modelEAST, modelEASP, modelEASTWI, modelEASZT, modelEATWIGEO]

    for i, (m, seed) in enumerate(zip(models, seeds)):

        # Specify model and MCMC parameters
        model = MCMC(NUTS(m), num_warmup=4000, num_samples=1000, num_chains=4)

        # Run MCMC
        model.run(jax.random.PRNGKey(seed), **train_data)

        # Generate Posterior Predictive Samples
        predictive = Predictive(m, posterior_samples=model.get_samples())
        post_pred_samples = predictive(jax.random.PRNGKey(seed),  **train_data)

        # Store samples
        results.append(model.get_samples())

        # For post-processing with arviz use
        idata = az.from_numpyro(model,posterior_predictive=post_pred_samples, log_likelihood=True)
        predictions.append(idata)

    # Loo summary check: Compute LOO
    loos = []
    for idata in predictions:
        loo = compute_pooled_loo(idata, pointwise=True)
        loos.append(loo)

    compare_df = az.compare({m: l for m, l in zip(model_name, loos)})
    print(compare_df)
    compare_df.to_csv(export_directory / f"{im_dwbar_file}_LOOCompare_{period}.csv", index=True)

    # Save results
    vars = [
        ["LE", "gammaE"],
        ["LE", "gammaE", "LA"],
        ["LE", "gammaE", "LA", "LS", "w"],
        ["LE", "gammaE", "LA", "LS", "LZ", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LT", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LP", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LTWI", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LT", "LZ", "w", "wS", "wZ", "wT"],
        ["LE", "gammaE", "LA", "LTWI", "LGEO", "w", "wGEO", "wTWI"]
    ]

    df_param_samples = [pd.DataFrame(res)[var]
                            for res, var in zip(results, vars)]

    df_param_samples[0].to_csv(export_directory.joinpath('%s_E_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[1].to_csv(export_directory.joinpath('%s_EA_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[2].to_csv(export_directory.joinpath('%s_EAS_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[3].to_csv(export_directory.joinpath('%s_EASZ_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[4].to_csv(export_directory.joinpath('%s_EAST_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[5].to_csv(export_directory.joinpath('%s_EASP_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[6].to_csv(export_directory.joinpath('%s_EASTWI_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[7].to_csv(export_directory.joinpath('%s_EASZT_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[8].to_csv(export_directory.joinpath('%s_EATWIGEO_%s.csv'%(im_dwbar_file,period)), index=False)

    # compute likelihoods
    # create test dataset
    Xtest = []; ztest = []
    for eqid in test_evid:
      group = im_dwbar[im_dwbar.evid==eqid].copy()
      Xtest.append(group[['epicenter_dist','epicenter_azimuth','vs30','Z1.0','TRI','TWI','geoage','precip']].values)
      ztest.append(group[period].values)
    test_data = dict(X=Xtest, eqids=test_evid, z=ztest)

    # event specific LPDDs
    loglik = [pd.DataFrame(log_likelihood(m, res, **test_data))
                for res, m in zip(results, models)]

    loglik[0].to_csv(export_directory.joinpath('%s_LPPDE_E_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[1].to_csv(export_directory.joinpath('%s_LPPDE_EA_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[2].to_csv(export_directory.joinpath('%s_LPPDE_EAS_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[3].to_csv(export_directory.joinpath('%s_LPPDE_EASZ_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[4].to_csv(export_directory.joinpath('%s_LPPDE_EAST_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[5].to_csv(export_directory.joinpath('%s_LPPDE_EASP_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[6].to_csv(export_directory.joinpath('%s_LPPDE_EASTWI_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[7].to_csv(export_directory.joinpath('%s_LPPDE_EASZT_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[8].to_csv(export_directory.joinpath('%s_LPPDE_EATWIGEO_%s.csv'%(im_dwbar_file,period)), index=False)

    # Pooled models LPPD
    n_r = len(df_param_samples[0])
    loglik_tot = [np.sum(LL.values, axis=1) for LL in loglik]
    LPPD = [-np.log(n_r) + logsumexp(LL) for LL in loglik_tot]

    ztot = np.hstack(test_data['z'])
    LPPD_ind = (-0.5 * (len(ztot)*np.log(2*np.pi) + np.sum(ztot**2)))
    rel_diff = [(1-lppd/LPPD_ind)*100 for lppd in LPPD]

    loglikpool = pd.DataFrame({
            'model': model_name,
            'period': [period] * 9,
            'LPPD': LPPD,
            'rel_diff': rel_diff
    })
    pooled_summary_all.append(loglikpool)

    # parameter summary
    period_summary = {"period": period}
    for model_name_i, var_list, res in zip(model_name, vars, results):
        df = pd.DataFrame(res)[var_list]
        for col in df.columns:
            period_summary[f"{model_name_i}_{col}"] = df[col].mean()
            period_summary[f"{model_name_i}_{col}_med"] = df[col].median()
            period_summary[f"{model_name_i}_{col}_5"] = df[col].quantile(0.05)
            period_summary[f"{model_name_i}_{col}_95"] = df[col].quantile(0.95)
    period_summaries.append(period_summary)

df_period_summary = pd.DataFrame(period_summaries)
df_period_summary.to_csv(export_directory / f"{im_dwbar_file}_param_summary.csv", index=False)

LPPDP = pd.concat(pooled_summary_all, ignore_index=True)
LPPDP.to_csv(export_directory / f"{im_dwbar_file}_LPPDP.csv", index=False)

# 5.0 Per Period Model fitting

## 5.1 Specify Inputs

In [ ]:
# specify filename prefix
im_dwbar_file = 'NZGMDB_v4p3_dwbar'

# input/output directory
input_directory = RESIDUALS_DIR
export_directory = SCM_DIR
export_directory.mkdir(parents=True, exist_ok=True)

# specify periods and prefix
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']
pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

## 5.2 Run Analysis

In [ ]:
# Define random number and models
rng = np.random.default_rng(31)
model_name = ['modelE', 'modelEA', 'modelEAS', 'modelEASZ', 'modelEAST','modelEASZT']

pooled_summary_all = []
period_summaries = []

for pID, period in enumerate(pSAobs_T):
    clear_output(wait=True)
    print(f"Period: {period}")

    #load file
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    evid_40 = im_dwbar['evid'].unique().tolist()

    # Specify evids from pooled dataset
    eqids = evid_40

    # create training dataset
    Xtot = []; ztot = []
    for eqid in eqids:
      group = im_dwbar[im_dwbar.evid==eqid].copy()
      Xtot.append(group[['epicenter_dist','epicenter_azimuth','vs30','Z1.0','TRI','TWI','geoage']].values)
      ztot.append(group[period].values)
    train_data = dict(X=Xtot, eqids=eqids, z=ztot)

    # generate new seed per period
    seeds = rng.integers(0, 100000, 6)

    results = []
    predictions = []
    lppds = []
    models = [modelE, modelEA, modelEAS, modelEASZ, modelEAST, modelEASZT]

    for i, (m, seed) in enumerate(zip(models, seeds)):

        # Specify model and MCMC parameters
        model = MCMC(NUTS(m), num_warmup=4000, num_samples=1000, num_chains=4)
        # Run MCMC
        model.run(jax.random.PRNGKey(seed), **train_data)

        # Generate Posterior Predictive Samples
        predictive = Predictive(m, posterior_samples=model.get_samples())
        post_pred_samples = predictive(jax.random.PRNGKey(seed),  **train_data)

        # Store samples
        results.append(model.get_samples())

        # For post-processing with arviz use
        idata = az.from_numpyro(model,posterior_predictive=post_pred_samples, log_likelihood=True)
        predictions.append(idata)

    # Loo summary check
    # Compute LOO
    loos = []
    for idata in predictions:
        loo = compute_pooled_loo(idata, pointwise=True)
        loos.append(loo)

    compare_df = az.compare({m: l for m, l in zip(model_name, loos)})
    print(compare_df)
    compare_df.to_csv(export_directory / f"{im_dwbar_file}_LOOCompare_{period}.csv", index=True)

    # Save results
    vars = [
        ["LE", "gammaE"],
        ["LE", "gammaE", "LA"],
        ["LE", "gammaE", "LA", "LS", "w"],
        ["LE", "gammaE", "LA", "LS", "LZ", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LT", "w", "wS"],
        ["LE", "gammaE", "LA", "LS", "LT", "LZ", "w", "wS", "wZ", "wT"]
    ]

    df_param_samples = [pd.DataFrame(res)[var]
                            for res, var in zip(results, vars)]

    df_param_samples[0].to_csv(export_directory.joinpath('%s_E_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[1].to_csv(export_directory.joinpath('%s_EA_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[2].to_csv(export_directory.joinpath('%s_EAS_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[3].to_csv(export_directory.joinpath('%s_EASZ_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[4].to_csv(export_directory.joinpath('%s_EAST_%s.csv'%(im_dwbar_file,period)), index=False)
    df_param_samples[5].to_csv(export_directory.joinpath('%s_EASZT_%s.csv'%(im_dwbar_file,period)), index=False)

    # compute likelihoods
    # event specific LPDDs
    loglik = [pd.DataFrame(log_likelihood(m, res, **train_data))
                for res, m in zip(results, models)]

    loglik[0].to_csv(export_directory.joinpath('%s_LPPDE_E_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[1].to_csv(export_directory.joinpath('%s_LPPDE_EA_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[2].to_csv(export_directory.joinpath('%s_LPPDE_EAS_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[3].to_csv(export_directory.joinpath('%s_LPPDE_EASZ_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[4].to_csv(export_directory.joinpath('%s_LPPDE_EAST_%s.csv'%(im_dwbar_file,period)), index=False)
    loglik[5].to_csv(export_directory.joinpath('%s_LPPDE_EASZT_%s.csv'%(im_dwbar_file,period)), index=False)

    # Pooled models
    n_r = len(df_param_samples[0])
    loglik_tot = [np.sum(LL.values, axis=1) for LL in loglik]
    LPPD = [-np.log(n_r) + logsumexp(LL) for LL in loglik_tot]

    ztot = np.hstack(train_data['z'])
    LPPD_ind = (-0.5 * (len(ztot)*np.log(2*np.pi)
                        + np.sum(ztot**2)))

    rel_diff = [(1-lppd/LPPD_ind)*100 for lppd in LPPD]

    loglikpool = pd.DataFrame({
            'model': model_name,
            'period': [period] * 6,
            'LPPD': LPPD,
            'rel_diff': rel_diff
    })
    pooled_summary_all.append(loglikpool)

    # parameter summary
    period_summary = {"period": period}
    for model_name_i, var_list, res in zip(model_name, vars, results):
        df = pd.DataFrame(res)[var_list]
        for col in df.columns:
            period_summary[f"{model_name_i}_{col}"] = df[col].mean()
            period_summary[f"{model_name_i}_{col}_med"] = df[col].median()
            period_summary[f"{model_name_i}_{col}_5"] = df[col].quantile(0.05)
            period_summary[f"{model_name_i}_{col}_95"] = df[col].quantile(0.95)
    period_summaries.append(period_summary)

df_period_summary = pd.DataFrame(period_summaries)
df_period_summary.to_csv(export_directory / f"{im_dwbar_file}_param_summary.csv", index=False)

LPPDP = pd.concat(pooled_summary_all, ignore_index=True)
LPPDP.to_csv(export_directory / f"{im_dwbar_file}_LPPDP.csv", index=False)

# 6.0 Model E event-wise per period fitting

## 6.1 Specify Inputs

In [ ]:
# specify filename prefix
im_dwbar_file = 'NZGMDB_v4p3_dwbar'

# input/output directory
input_directory = RESIDUALS_DIR
export_directory = SCM_DIR
export_directory.mkdir(parents=True, exist_ok=True)

# specify periods and prefix
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']
pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

## 6.2 Run Analysis

In [ ]:
rng = np.random.default_rng(91)
seeds = rng.integers(0, 100000, len(evid_40))

for pID, period in enumerate(pSAobs_T):

    # lists for storage on each loop
    lppds = []
    LE = []
    LE95 = []
    gE = []
    gE95 = []

    clear_output(wait=True)
    print(f"Period: {period}")

    #load file
    im_dwbar = pd.read_csv(input_directory / f"{im_dwbar_file}_{pSA_names[pID]}.csv", dtype={"sta": "string"})
    evid_40 = im_dwbar['evid'].unique().tolist()
    n_recs = im_dwbar['evid'].value_counts().rename('n_recs')

    # dataframe for storage
    df_es = pd.DataFrame()
    df_es = pd.DataFrame({'evid': evid_40})
    df_es = df_es.merge(n_recs, left_on='evid', right_index=True)

    for i, eqid in enumerate(evid_40):

        clear_output(wait=True)  # Clears previous output (nice for progress updates)
        print(f"Processing event {i+1}/{len(evid_40)}: {eqid}, Period: {period}")

        seed = seeds[i]

        # create dataset
        Xtot = []; ztot = []
        group = im_dwbar[im_dwbar.evid==eqid].copy()
        X = [group[['epicenter_dist', 'epicenter_azimuth', 'vs30']].values]
        z = [group[period].values]
        train_data = dict(X=X, eqids=[eqid], z=z)

        # Specify model and MCMC parameters
        model = MCMC(NUTS(modelE), num_warmup=4000, num_samples=1000, num_chains=4)
        # Run MCMC
        model.run(jax.random.PRNGKey(seed), **train_data)

        # Generate Posterior Predictive Samples
        predictive = Predictive(modelE, posterior_samples=model.get_samples())
        post_pred_samples = predictive(jax.random.PRNGKey(seed),  **train_data)

        # Store samples
        samples = model.get_samples()
        LE.append(np.mean(samples['LE']))
        LE95.append(np.percentile(samples['LE'], 95))
        gE.append(np.mean(samples['gammaE']))
        gE95.append(np.percentile(samples['gammaE'], 95))

        # Compute log-likelihood for each sample
        cond_loglik = log_likelihood(modelE, samples, **train_data)['z_' + str(eqid)]

        # Compute Log-posterior predictive (LPPD)
        lppds.append(-np.log(4000) + logsumexp(jax.device_get(cond_loglik)))

    df_es[period] = lppds
    df_es[period + '_LE'] = LE
    df_es[period + '_LE95'] = LE95
    df_es[period + '_gE'] = gE
    df_es[period + '_gE95'] = gE95
    df_es.to_csv(export_directory / f"{im_dwbar_file}_{pSA_names[pID]}_LPPDPES.csv", index=False)